### **Librerías**

In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
from skforecast.recursive import ForecasterRecursive
from prophet import Prophet
import joblib
import warnings

warnings.filterwarnings("ignore")

### **Parámetros**

In [ ]:
os.chdir("/Users/juanchacon/Library/Mobile Documents/com~apple~CloudDocs/GitHub/bot_futuros")

SILVER_DATA_PATH = os.getcwd() + "/data/silver"
RESULTS_PATH = os.getcwd() + "/notebooks/EDA/results_preds"
FILE_PATH = f"{SILVER_DATA_PATH}/datos_DEMANDA_COMPRADOR.csv"
FORECAST_HORIZON_DAYS = 180  # 6 meses aprox
TRAIN_SIZE_PROP = 0.8  # Proporción del dataset para entrenar el modelo

### **Funciones**

In [16]:
# ==========================================
# 1. FUNCIÓN DE CARGA DE DATOS
# ==========================================

def load_data(filepath):
    print(f"Cargando datos desde: {filepath}...")
    df = pd.read_csv(filepath)
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df = df.set_index('Fecha')
    df = df.asfreq('D')
    df = df.fillna(method='ffill')
    return df

# ==========================================
# 2. FUNCIONES DE MÉTRICAS
# ==========================================
def calculate_metrics(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    me = np.mean(y_true - y_pred)
    
    return {
        'Model': model_name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'ME': me
    }

# ==========================================
# 3. MODELOS INDIVIDUALES
# ==========================================
def model_moving_average(train, test_steps, window=7):
    last_window_mean = train.iloc[-window:].mean()
    pred = pd.Series([last_window_mean] * test_steps, 
                     index=train.index[-1] + pd.to_timedelta(np.arange(1, test_steps + 1), unit='D'))
    return pred, window

def model_simple_exp_smoothing(train, test_steps):
    model = SimpleExpSmoothing(train).fit(optimized=True)
    pred = model.forecast(test_steps)
    return pred, model

def model_double_exp_smoothing(train, test_steps):
    model = ExponentialSmoothing(train, trend='add', seasonal=None).fit(optimized=True)
    pred = model.forecast(test_steps)
    return pred, model

def model_holt_winters(train, test_steps):
    # Estacionalidad semanal (7 días)
    model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7).fit(optimized=True)
    pred = model.forecast(test_steps)
    return pred, model

def model_arima(train, test_steps):
    model = auto_arima(train, seasonal=True, m=7, stepwise=True, suppress_warnings=True, error_action='ignore')
    pred = model.predict(n_periods=test_steps)
    pred = pd.Series(pred, index=train.index[-1] + pd.to_timedelta(np.arange(1, test_steps + 1), unit='D'))
    return pred, model

def model_prophet(train, test_steps):
    df_train = train.reset_index().rename(columns={train.index.name: 'ds', train.name: 'y'})
    m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
    m.fit(df_train)
    future = m.make_future_dataframe(periods=test_steps)
    forecast = m.predict(future)
    pred = forecast.iloc[-test_steps:]['yhat']
    pred.index = train.index[-1] + pd.to_timedelta(np.arange(1, test_steps + 1), unit='D')
    return pred, m

def model_skforecast(train, test_steps):
    forecaster = ForecasterRecursive(
        estimator = RandomForestRegressor(random_state=123, n_estimators=100),
        lags = 7
    )
    
    forecaster.fit(y=train)
    pred = forecaster.predict(steps=test_steps)
    return pred, forecaster

# ==========================================
# 4. ENSAMBLE OPTIMIZADO
# ==========================================
def optimize_ensemble_weights(predictions_dict, y_true):
    """
    Encuentra los pesos óptimos para minimizar el MSE usando programación cuadrática (SLSQP).
    """
    model_names = list(predictions_dict.keys())
    preds_matrix = np.array([predictions_dict[name].values for name in model_names]).T
    y_true_vals = y_true.values
    
    # Función objetivo: Minimizar MSE
    def loss_function(weights):
        # Predicción combinada = suma(peso_i * prediccion_i)
        final_pred = np.dot(preds_matrix, weights)
        return mean_squared_error(y_true_vals, final_pred)
    
    # Restricción: La suma de pesos debe ser 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    
    # Límites: Los pesos deben estar entre 0 y 1
    bounds = tuple((0, 1) for _ in range(len(model_names)))
    
    # Inicialización: Pesos iguales
    initial_weights = [1/len(model_names)] * len(model_names)
    
    # Optimización
    result = minimize(loss_function, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)
    
    optimal_weights = dict(zip(model_names, result.x))
    
    # Calcular la predicción final con los pesos óptimos
    final_pred_values = np.dot(preds_matrix, result.x)
    ensemble_series = pd.Series(final_pred_values, index=y_true.index)
    
    return ensemble_series, optimal_weights

# ==========================================
# 5. PIPELINE PRINCIPAL
# ==========================================
def run_forecast_pipeline(file_path=FILE_PATH):
    df = load_data(file_path)
    target_columns = df.columns
    results_summary = []
    prediction_storage = {} # Para guardar las predicciones de cada modelo por columna
    
    for col in target_columns:
        print(f"\n{'='*50}")
        print(f" PROCESANDO COLUMNA: {col}")
        print(f"{'='*50}")
        
        series = df[col]
        
        # --- SPLIT 80-20 ---
        train_size = int(len(series) * TRAIN_SIZE_PROP)
        train = series.iloc[:train_size]
        test = series.iloc[train_size:]
        
        print(f"Datos totales: {len(series)} | Train: {len(train)} | Test: {len(test)}")
        
        predictions = {}
        
        # 1. Ejecutar Modelos Base
        try:
            print(" -> Entrenando Moving Average...")
            predictions['Moving_Avg'], _ = model_moving_average(train, len(test))
            print(" -> Entrenando Simple Exponential Smoothing...")
            predictions['Simple_Exp'], _ = model_simple_exp_smoothing(train, len(test))
            print(" -> Entrenando Double Exponential Smoothing...")
            predictions['Double_Exp'], _ = model_double_exp_smoothing(train, len(test))
            print(" -> Entrenando Holt-Winters...")
            predictions['Holt_Winters'], _ = model_holt_winters(train, len(test))
            print(" -> Entrenando ARIMA...")
            predictions['ARIMA'], _ = model_arima(train, len(test))
            print(" -> Entrenando Prophet...")
            predictions['Prophet'], _ = model_prophet(train, len(test))
            print(" -> Entrenando Skforecast (Random Forest)...")
            predictions['Skforecast'], _ = model_skforecast(train, len(test))
        except Exception as e:
            print(f"Error en entrenamiento base: {e}")
            import traceback
            traceback.print_exc()
            continue

        # 2. Optimizar Ensamble
        print(" -> Optimizando pesos del Ensamble...")
        predictions['Ensemble'], ensemble_weights = optimize_ensemble_weights(predictions, test)
        
        # Imprimir pesos encontrados
        print(" Pesos óptimos encontrados:")
        for m, w in ensemble_weights.items():
            if w > 0.001: # Solo mostrar pesos relevantes
                print(f"   - {m}: {w:.4f}")

        # 3. Evaluación
        col_metrics = []
        best_model_name = ""
        min_mse = float('inf')
        
        print("\n--- Resultados de Métricas (Test Set) ---")
        for name, pred in predictions.items():
            metrics = calculate_metrics(test, pred, name)
            col_metrics.append(metrics)
            print(f"{name}: MSE={metrics['MSE']:.2f}, MAPE={metrics['MAPE']:.4f}")
            
            if metrics['MSE'] < min_mse:
                min_mse = metrics['MSE']
                best_model_name = name
        
        print(f"\n>> MEJOR MODELO PARA {col}: {best_model_name}")
        
        # 4. Gráfico de Validación
        plt.figure(figsize=(14, 7))
        plt.plot(train.index[-400:], train[-400:], label='Train (Zoom)', color='gray', alpha=0.5)
        plt.plot(test.index, test, label='Real (Test)', color='black', linewidth=2)
        
        # Graficar solo el mejor y el ensamble para no saturar, o todos si se prefiere
        for name, pred in predictions.items():
            if name == best_model_name:
                plt.plot(test.index, pred, label=f'{name} (Best)', linewidth=2.5)
            elif name == 'Ensemble':
                plt.plot(test.index, pred, label='Ensemble', linestyle='--', linewidth=2)
            else:
                # Opcional: Graficar los otros con transparencia alta
                plt.plot(test.index, pred, alpha=0.2, color='grey')

        plt.title(f'Validación - {col} (Split 80/20)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(f'{RESULTS_PATH}/validacion_{col}.png')
        plt.close() # Cerrar para no consumir memoria
        
        # 5. Re-entrenamiento con TODO el dataset y Pronóstico Futuro
        print(f" Generando pronóstico a 6 meses ({FORECAST_HORIZON_DAYS} días)...")
        
        final_forecast = None
        
        # Reentrenamos modelos necesarios para el futuro
        # Si el mejor es Ensemble, necesitamos reentrenar TODOS sus componentes
        if best_model_name == 'Ensemble':
            future_preds = {}
            # Reentrenar todos con data completa
            p_ma, _ = model_moving_average(series, FORECAST_HORIZON_DAYS)
            future_preds['Moving_Avg'] = p_ma
            
            _, m_ses = model_simple_exp_smoothing(series, FORECAST_HORIZON_DAYS)
            future_preds['Simple_Exp'] = m_ses.forecast(FORECAST_HORIZON_DAYS)
            
            _, m_des = model_double_exp_smoothing(series, FORECAST_HORIZON_DAYS)
            future_preds['Double_Exp'] = m_des.forecast(FORECAST_HORIZON_DAYS)
            
            _, m_hw = model_holt_winters(series, FORECAST_HORIZON_DAYS)
            future_preds['Holt_Winters'] = m_hw.forecast(FORECAST_HORIZON_DAYS)
            
            _, m_arima = model_arima(series, FORECAST_HORIZON_DAYS)
            future_preds['ARIMA'] = pd.Series(m_arima.predict(n_periods=FORECAST_HORIZON_DAYS), index=p_ma.index)
            
            _, m_prophet = model_prophet(series, FORECAST_HORIZON_DAYS)
            fut = m_prophet.make_future_dataframe(periods=FORECAST_HORIZON_DAYS)
            future_preds['Prophet'] = m_prophet.predict(fut).iloc[-FORECAST_HORIZON_DAYS:]['yhat']
            future_preds['Prophet'].index = p_ma.index
            
            _, m_sk = model_skforecast(series, FORECAST_HORIZON_DAYS)
            future_preds['Skforecast'] = m_sk.predict(steps=FORECAST_HORIZON_DAYS)
            
            # Combinar usando los pesos optimizados calculados previamente
            final_vals = np.zeros(FORECAST_HORIZON_DAYS)
            for m_name, weight in ensemble_weights.items():
                if m_name in future_preds:
                    final_vals += future_preds[m_name].values * weight
            
            final_forecast = pd.Series(final_vals, index=p_ma.index)
            joblib.dump(ensemble_weights, f'{RESULTS_PATH}/modelo_{col}_ensemble_weights.pkl')
            
        else:
            # Si ganó un modelo individual, reentrenamos solo ese
            if best_model_name == 'Moving_Avg':
                final_forecast, window = model_moving_average(series, FORECAST_HORIZON_DAYS)
                with open(f'{RESULTS_PATH}/modelo_{col}_moving_avg_window.txt', 'w') as f:
                    f.write(f"Window: {window}")

            elif best_model_name == 'Simple_Exp':
                _, m = model_simple_exp_smoothing(series, FORECAST_HORIZON_DAYS)
                final_forecast = m.forecast(FORECAST_HORIZON_DAYS)
                joblib.dump(m, f'{RESULTS_PATH}/modelo_{col}.pkl')
            elif best_model_name == 'Double_Exp':
                _, m = model_double_exp_smoothing(series, FORECAST_HORIZON_DAYS)
                final_forecast = m.forecast(FORECAST_HORIZON_DAYS)
                joblib.dump(m, f'{RESULTS_PATH}/modelo_{col}.pkl')
            elif best_model_name == 'Holt_Winters':
                _, m = model_holt_winters(series, FORECAST_HORIZON_DAYS)
                final_forecast = m.forecast(FORECAST_HORIZON_DAYS)
                joblib.dump(m, f'{RESULTS_PATH}/modelo_{col}.pkl')
            elif best_model_name == 'ARIMA':
                _, m = model_arima(series, FORECAST_HORIZON_DAYS)
                final_forecast = m.predict(n_periods=FORECAST_HORIZON_DAYS)
                joblib.dump(m, f'{RESULTS_PATH}/modelo_{col}.pkl')
            elif best_model_name == 'Prophet':
                _, m = model_prophet(series, FORECAST_HORIZON_DAYS)
                fut = m.make_future_dataframe(periods=FORECAST_HORIZON_DAYS)
                final_forecast = m.predict(fut).iloc[-FORECAST_HORIZON_DAYS:]['yhat']
                joblib.dump(m, f'{RESULTS_PATH}/modelo_{col}.pkl')
            elif best_model_name == 'Skforecast':
                _, m = model_skforecast(series, FORECAST_HORIZON_DAYS)
                final_forecast = m.predict(steps=FORECAST_HORIZON_DAYS)
                joblib.dump(m, f'{RESULTS_PATH}/modelo_{col}.pkl')

        # Guardar CSV Futuro
        if isinstance(final_forecast, pd.Series) or isinstance(final_forecast, pd.DataFrame):
            df_futuro = pd.DataFrame({'Fecha': final_forecast.index, 'Prediccion': final_forecast.values})
            prediction_storage[col] = df_futuro

        # Guardar métricas del mejor
        best_metrics = next(m for m in col_metrics if m['Model'] == best_model_name)
        results_summary.append({
            'Columna': col,
            'Mejor_Modelo': best_model_name,
            'MSE': best_metrics['MSE'],
            'RMSE': best_metrics['RMSE'],
            'MAPE': best_metrics['MAPE']
        })

    # Resumen Final
    summary_df = pd.DataFrame(results_summary)
    print("\nPredicciones finalizadas exitosamente.")

    return summary_df, prediction_storage

In [17]:
summary_df, prediction_storage = run_forecast_pipeline(FILE_PATH)

Cargando datos desde: /Users/juanchacon/Library/Mobile Documents/com~apple~CloudDocs/GitHub/bot_futuros/data/silver/datos_DEMANDA_COMPRADOR.csv...

 PROCESANDO COLUMNA: Demanda_kWh_0-7
Datos totales: 771 | Train: 616 | Test: 155
 -> Entrenando Moving Average...
 -> Entrenando Simple Exponential Smoothing...
 -> Entrenando Double Exponential Smoothing...
 -> Entrenando Holt-Winters...
 -> Entrenando ARIMA...


15:08:54 - cmdstanpy - INFO - Chain [1] start processing
15:08:54 - cmdstanpy - INFO - Chain [1] done processing


 -> Entrenando Prophet...
 -> Entrenando Skforecast (Random Forest)...


15:08:54 - cmdstanpy - INFO - Chain [1] start processing
15:08:55 - cmdstanpy - INFO - Chain [1] done processing


 -> Optimizando pesos del Ensamble...
 Pesos óptimos encontrados:
   - Moving_Avg: 0.1426
   - Simple_Exp: 0.1426
   - Double_Exp: 0.1443
   - Holt_Winters: 0.1426
   - ARIMA: 0.1426
   - Prophet: 0.1443
   - Skforecast: 0.1426

--- Resultados de Métricas (Test Set) ---
Moving_Avg: MSE=204801127813.89, MAPE=0.0879
Simple_Exp: MSE=157576344101.07, MAPE=0.0765
Double_Exp: MSE=94279257345.34, MAPE=0.0596
Holt_Winters: MSE=131910204713.61, MAPE=0.0698
ARIMA: MSE=133109993525.28, MAPE=0.0711
Prophet: MSE=83665488485.95, MAPE=0.0571
Skforecast: MSE=164535730381.81, MAPE=0.0781
Ensemble: MSE=100243106007.57, MAPE=0.0624

>> MEJOR MODELO PARA Demanda_kWh_0-7: Prophet
 Generando pronóstico a 6 meses (180 días)...

 PROCESANDO COLUMNA: Demanda_kWh_7-17
Datos totales: 771 | Train: 616 | Test: 155
 -> Entrenando Moving Average...
 -> Entrenando Simple Exponential Smoothing...
 -> Entrenando Double Exponential Smoothing...
 -> Entrenando Holt-Winters...
 -> Entrenando ARIMA...


...

 -> Optimizando pesos del Ensamble...
 Pesos óptimos encontrados:
   - Prophet: 1.0000

--- Resultados de Métricas (Test Set) ---
Moving_Avg: MSE=181206139391.68, MAPE=0.0950
Simple_Exp: MSE=147102828507.37, MAPE=0.0842
Double_Exp: MSE=102479479802.26, MAPE=0.0705
Holt_Winters: MSE=123210488914.91, MAPE=0.0768
ARIMA: MSE=121321596468.38, MAPE=0.0764
Prophet: MSE=54850201887.69, MAPE=0.0496
Skforecast: MSE=128216901440.05, MAPE=0.0786
Ensemble: MSE=54850201887.69, MAPE=0.0496

>> MEJOR MODELO PARA Demanda_kWh_17-23: Prophet
 Generando pronóstico a 6 meses (180 días)...

 PROCESANDO COLUMNA: Demanda_kWh_Dia
Datos totales: 771 | Train: 616 | Test: 155
 -> Entrenando Moving Average...
 -> Entrenando Simple Exponential Smoothing...
 -> Entrenando Double Exponential Smoothing...
 -> Entrenando Holt-Winters...
 -> Entrenando ARIMA...


15:09:26 - cmdstanpy - INFO - Chain [1] start processing
15:09:26 - cmdstanpy - INFO - Chain [1] done processing


 -> Entrenando Prophet...
 -> Entrenando Skforecast (Random Forest)...


15:09:27 - cmdstanpy - INFO - Chain [1] start processing
15:09:27 - cmdstanpy - INFO - Chain [1] done processing


 -> Optimizando pesos del Ensamble...
 Pesos óptimos encontrados:
   - Moving_Avg: 0.1428
   - Simple_Exp: 0.1428
   - Double_Exp: 0.1429
   - Holt_Winters: 0.1429
   - ARIMA: 0.1428
   - Prophet: 0.1429
   - Skforecast: 0.1428

--- Resultados de Métricas (Test Set) ---
Moving_Avg: MSE=1818128528356.06, MAPE=0.0903
Simple_Exp: MSE=1228016488468.25, MAPE=0.0746
Double_Exp: MSE=761622219893.63, MAPE=0.0595
Holt_Winters: MSE=994395846411.66, MAPE=0.0672
ARIMA: MSE=1424081906793.68, MAPE=0.0800
Prophet: MSE=759787546228.72, MAPE=0.0590
Skforecast: MSE=3023494105641.03, MAPE=0.1220
Ensemble: MSE=1020583028295.84, MAPE=0.0686

>> MEJOR MODELO PARA Demanda_kWh_Dia: Prophet
 Generando pronóstico a 6 meses (180 días)...

Predicciones finalizadas exitosamente.
